In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType


In [0]:
input_path = "/Volumes/workspace/recruitment/policies/input"
checkpoint_path = "/Volumes/workspace/recruitment/policies/checkpoints/bronze_claims"
bronze_table = "recruitment.bronze_claims"

In [0]:

bronze_schema = StructType([
    StructField(column_name, StringType(), True)
    for column_name in [
        "months_as_customer",
        "age",
        "policy_number",
        "policy_bind_date",
        "policy_state",
        "policy_csl",
        "policy_deductable",
        "policy_annual_premium",
        "umbrella_limit",
        "insured_zip",
        "insured_sex",
        "insured_education_level",
        "insured_occupation",
        "insured_hobbies",
        "insured_relationship",
        "capital-gains",
        "capital-loss",
        "incident_date",
        "incident_type",
        "collision_type",
        "incident_severity",
        "authorities_contacted",
        "incident_state",
        "incident_city",
        "incident_location",
        "incident_hour_of_the_day",
        "number_of_vehicles_involved",
        "property_damage",
        "bodily_injuries",
        "witnesses",
        "police_report_available",
        "total_claim_amount",
        "injury_claim",
        "property_claim",
        "vehicle_claim",
        "auto_make",
        "auto_model",
        "auto_year",
        "fraud_reported",
        "timestamp",
    ]
 ])

In [0]:
stream_df = (
    spark.readStream
    .option("header", True)
    .option("delimiter", ";")
    .schema(bronze_schema)
    .csv(input_path)
)

In [0]:
bronze_df = (
    stream_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

In [0]:
display(spark.table(bronze_table))
print(spark.table(bronze_table).count())
